In [1]:
# you will be prompted with a window asking to grant permissions
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [2]:
!mkdir -p /content/computer_vision
!find "/content/drive/Shareddrives/Computer Vision/output/" -mindepth 1 -maxdepth 1 -exec cp -r -t /content/computer_vision/ {} +


In [3]:
# If you need to install anything in notebooks, uncomment:
# %pip install torch torchvision torchaudio --upgrade
# %pip install scikit-learn

import os, json, math, random
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet34_Weights

print("Torch:", torch.__version__)
print("Torchvision:", models.__version__ if hasattr(models, "__version__") else "n/a")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Torch: 2.9.0+cu126
Torchvision: n/a
Using device: cuda


In [4]:
@dataclass
class CFG:
    data_root: str = "/content/computer_vision/"        # expects output/train, output/val (optional), output/test (optional)
    num_classes: int = 10            # your 10 face types
    img_size: int = 32
    batch_size: int = 32
    num_workers: int = 4
    seed: int = 42

    # Stage 1 (freeze backbone), then Stage 2 (unfreeze full model)
    epochs_stage1: int = 10
    epochs_stage2: int = 20
    lr_head: float = 1e-3
    lr_all: float = 1e-4
    weight_decay: float = 1e-4

    # Early stopping (Stage 2)
    early_stopping_patience: int = 7

    # Checkpoints
    ckpt_dir: str = "checkpoints"
    # Save persistently to Shared Drive
    drive_save_dir: str = "/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints"
    # ckpt_name: str = "resnet50_faces_best_32.pt"
    ckpt_name: str = "resnet34_faces_best_32.pt"

cfg = CFG()
print(f"Config configured. Model checkpoints will be saved to: {cfg.drive_save_dir}")

Config configured. Model checkpoints will be saved to: /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints


In [5]:
import shutil
from pathlib import Path

# Source paths (local)
ckpt_name = "resnet34_faces_best_32.pt"
local_ckpt = Path("checkpoints") / ckpt_name
local_label_map = Path("checkpoints/label_map.json")

# Destination paths (Shared Drive)
dest_dir = Path("/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints")
dest_dir.mkdir(parents=True, exist_ok=True)

# Copy Model
if local_ckpt.exists():
    shutil.copy(local_ckpt, dest_dir / ckpt_name)
    print(f"Successfully copied model to: {dest_dir / ckpt_name}")
else:
    print(f"Error: Local model not found at {local_ckpt}")

# Copy Label Map
if local_label_map.exists():
    shutil.copy(local_label_map, dest_dir / "label_map.json")
    print(f"Successfully copied label map to: {dest_dir / 'label_map.json'}")
else:
    print(f"Warning: Local label map not found at {local_label_map}")

Error: Local model not found at checkpoints/resnet34_faces_best_32.pt


In [6]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_by_class(dataset: datasets.ImageFolder, indices=None):
    """Count examples per class. If indices are given, count only those."""
    if indices is None:
        targets = [dataset.samples[i][1] for i in range(len(dataset.samples))]
    else:
        targets = [dataset.samples[i][1] for i in indices]
    counts = [0] * len(dataset.classes)
    for t in targets:
        counts[t] += 1
    return counts

def compute_class_weights(counts):
    """Inverse-frequency weights normalized like N/(K*n_i)."""
    total = sum(counts)
    K = len(counts)
    weights = [total / (K * c) if c > 0 else 0.0 for c in counts]
    return torch.tensor(weights, dtype=torch.float)

def save_label_map_from_classes(classes, path: Path):
    label_map = {int(i): cls for i, cls in enumerate(classes)}
    path.write_text(json.dumps(label_map, indent=2), encoding="utf-8")


In [7]:
# Cell X — ImageNet stats fallback (add this above the dataloaders cell)

# Standard ImageNet normalization constants
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_imagenet_mean_std():
    """
    Return mean/std from weights.meta when available; otherwise fall back to
    the standard ImageNet stats.
    """
    try:
        w = ResNet34_Weights.DEFAULT # Changed IMAGENET1K_V2 to DEFAULT
        meta = getattr(w, "meta", {}) or {}
        mean = tuple(meta.get("mean", IMAGENET_MEAN))
        std  = tuple(meta.get("std", IMAGENET_STD))
        return mean, std
    except Exception:
        return IMAGENET_MEAN, IMAGENET_STD

In [13]:
from typing import Tuple, List, Optional

def stratified_split_indices(dataset: datasets.ImageFolder, val_ratio=0.15, seed=42) -> Tuple[List[int], List[int]]:

    try:
        from sklearn.model_selection import StratifiedShuffleSplit
        y = [dataset.samples[i][1] for i in range(len(dataset.samples))]
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
        train_idx, val_idx = next(sss.split([[0]]*len(y), y))
        return train_idx.tolist(), val_idx.tolist()
    except Exception:
        N = len(dataset.samples)
        idx = list(range(N))
        random.Random(seed).shuffle(idx)
        split = int(N * (1 - val_ratio))
        return idx[:split], idx[split:]

def make_dataloaders(cfg: CFG):
    weights = ResNet34_Weights.DEFAULT
    # mean, std = weights.meta["mean"], weights.meta["std"]
    mean, std = get_imagenet_mean_std()

    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(cfg.img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.15)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    root = Path(cfg.data_root)
    train_dir = root / "train"
    val_dir = root / "val"
    test_dir = root / "test"

    if not train_dir.exists():
        raise FileNotFoundError(f"Missing folder: {train_dir}")

    # Base for splitting & class indexing
    base_train = datasets.ImageFolder(train_dir, transform=None)

    if val_dir.exists():
        train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
        val_ds   = datasets.ImageFolder(val_dir,   transform=eval_tfms)
        train_idx = list(range(len(train_ds.samples)))
        val_idx   = list(range(len(val_ds.samples)))
        classes = train_ds.classes
        class_to_idx = train_ds.class_to_idx
        counts = count_by_class(train_ds)  # counts from full train set
    else:
        train_idx, val_idx = stratified_split_indices(base_train, val_ratio=0.15, seed=cfg.seed)
        train_full = datasets.ImageFolder(train_dir, transform=train_tfms)
        val_full   = datasets.ImageFolder(train_dir, transform=eval_tfms)  # same root, eval tfms
        train_ds = Subset(train_full, train_idx)
        val_ds   = Subset(val_full,   val_idx)
        classes = base_train.classes
        class_to_idx = base_train.class_to_idx
        counts = count_by_class(base_train, train_idx)

    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

    test_dl = None
    if test_dir.exists():
        test_ds = datasets.ImageFolder(test_dir, transform=eval_tfms)
        test_dl = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                             num_workers=cfg.num_workers, pin_memory=True)

    return train_dl, val_dl, test_dl, counts, classes, class_to_idx

In [14]:
def build_model(num_classes: int):
    weights = ResNet34_Weights.DEFAULT  # Changed IMAGENET1K_V2 to DEFAULT
    model = models.resnet34(weights=weights)

    # Modify the first conv layer for smaller input images
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Replace maxpool with identity to avoid aggressive downsampling for 32x32 images
    model.maxpool = nn.Identity()

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(1)
        total_correct += (preds == targets).sum().item()
        total_samples += images.size(0)

    return total_loss / max(1, total_samples), total_correct / max(1, total_samples)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, targets)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(1)
        total_correct += (preds == targets).sum().item()
        total_samples += images.size(0)

    return total_loss / max(1, total_samples), total_correct / max(1, total_samples)


In [16]:
import shutil

def run_training(cfg: CFG):
    set_seed(cfg.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = (dev.type == "cuda")

    train_dl, val_dl, test_dl, counts, classes, class_to_idx = make_dataloaders(cfg)
    print("Classes:", classes)
    print("Train counts:", counts)
    if len(classes) != cfg.num_classes:
        print(f"[WARN] Found {len(classes)} classes on disk; CFG.num_classes={cfg.num_classes}. "
              f"Proceeding with {len(classes)}.")
    num_classes = len(classes)

    # Save label map
    ckpt_dir = Path(cfg.ckpt_dir); ckpt_dir.mkdir(parents=True, exist_ok=True)
    label_map_path = ckpt_dir / "label_map.json"
    save_label_map_from_classes(classes, label_map_path)
    print(f"Saved label map to {label_map_path}")

    # Backup label map to Drive if configured
    if cfg.drive_save_dir:
        drive_dir = Path(cfg.drive_save_dir)
        print(f"Backup directory set to: {drive_dir}")
        drive_dir.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy(label_map_path, drive_dir / "label_map.json")
            print(f"Backed up label map to {drive_dir}")
        except Exception as e:
            print(f"Warning: Could not backup label map to Drive: {e}")

    # Loss with class weights (handles imbalance)
    class_weights = compute_class_weights(counts).to(dev)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Build model
    model = build_model(num_classes=num_classes).to(dev)

    # -------- Stage 1: freeze backbone, train head --------
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.")

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=cfg.lr_head, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev.type == "cuda"))

    best_val_loss = math.inf

    def save_best_model(loss):
        nonlocal best_val_loss
        best_val_loss = loss
        # Save locally
        torch.save({
            "model_state": model.state_dict(),
            "classes": classes,
            "class_to_idx": class_to_idx,
            "cfg": vars(cfg)
        }, ckpt_dir / cfg.ckpt_name)
        print("  Saved (new best on val)")

        # Backup to Drive
        if cfg.drive_save_dir:
            drive_dir = Path(cfg.drive_save_dir)
            try:
                shutil.copy(ckpt_dir / cfg.ckpt_name, drive_dir / cfg.ckpt_name)
                print(f"  Backed up to {drive_dir / cfg.ckpt_name}")
            except Exception as e:
                print(f"  Warning: Backup failed: {e}")

    for epoch in range(1, cfg.epochs_stage1 + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, scaler, dev)
        va_loss, va_acc = evaluate(model, val_dl, criterion, dev)
        print(f"[Stage1][Epoch {epoch:02d}] train_loss={tr_loss:.4f} acc={tr_acc:.3f} | "
              f"val_loss={va_loss:.4f} acc={va_acc:.3f}")

        if va_loss < best_val_loss:
            save_best_model(va_loss)

    # -------- Stage 2: unfreeze all, fine-tune --------
    for p in model.parameters():
        p.requires_grad = True

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr_all, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs_stage2)

    no_improve = 0
    for epoch in range(1, cfg.epochs_stage2 + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, scaler, dev)
        va_loss, va_acc = evaluate(model, val_dl, criterion, dev)
        scheduler.step()

        print(f"[Stage2][Epoch {epoch:02d}] train_loss={tr_loss:.4f} acc={tr_acc:.3f} | "
              f"val_loss={va_loss:.4f} acc={va_acc:.3f}")

        if va_loss < best_val_loss:
            save_best_model(va_loss)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= cfg.early_stopping_patience:
                print(f"Early stopping (no val improvement for {cfg.early_stopping_patience} epochs).")
                break

    best_path = Path(cfg.ckpt_dir) / cfg.ckpt_name
    print(f"Best model saved locally at: {best_path}")
    if cfg.drive_save_dir:
        print(f"Best model backed up to: {Path(cfg.drive_save_dir) / cfg.ckpt_name}")

    return best_path, classes, test_dl, criterion, dev

In [17]:
best_ckpt_path, classes, test_dl, criterion, dev = run_training(cfg)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Classes: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009']
Train counts: [898, 853, 983, 916, 944, 822, 848, 825, 847, 845]
Saved label map to checkpoints/label_map.json
Backup directory set to: /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints
Backed up label map to /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 204MB/s]
/tmp/ipython-input-4190361076.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dev.type == "cuda"))
/tmp/ipython-input-505752090.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[Stage1][Epoch 01] train_loss=2.1579 acc=0.222 | val_loss=1.9850 acc=0.290
  Saved (new best on val)
  Backed up to /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet34_faces_best_32.pt
[Stage1][Epoch 02] train_loss=2.0077 acc=0.283 | val_loss=1.9027 acc=0.317
  Saved (new best on val)
  Backed up to /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet34_faces_best_32.pt
[Stage1][Epoch 03] train_loss=1.9550 acc=0.304 | val_loss=1.9040 acc=0.321
[Stage1][Epoch 04] train_loss=1.9431 acc=0.309 | val_loss=1.9108 acc=0.325
[Stage1][Epoch 05] train_loss=1.9209 acc=0.317 | val_loss=1.8637 acc=0.336
  Saved (new best on val)
  Backed up to /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet34_faces_best_32.pt
[Stage1][Epoch 06] train_loss=1.9178 acc=0.319 | val_loss=1.8862 acc=0.329
[Stage1][Epoch 07] train_loss=1.9041 acc=0.322 | val_loss=1.8612 acc=0.335
  Saved (new best on val)
  Backed up to /content/drive/Shareddrives/C

In [18]:
# === Cell: Prepare classes, test_dl, criterion, dev (independent of run_training) ===
import os, json
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Dict, List

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

# ---------- 1) Config (reuses your cfg/CFG if present) ----------
@dataclass
class _EVAL_CFG:
    data_root: str = getattr(globals().get("cfg", object()), "data_root", "output")  # reuse cfg if defined
    img_size: int  = getattr(globals().get("cfg", object()), "img_size", 32)
    batch_size: int = getattr(globals().get("cfg", object()), "batch_size", 32)
    num_workers: int = getattr(globals().get("cfg", object()), "num_workers", 4)
    use_class_weights: bool = True
    # Updated to match the training config filename
    ckpt_path: str = "checkpoints/resnet34_faces_best_32.pt"

EVAL = _EVAL_CFG()

# ---------- 2) ImageNet mean/std (reuse your helper if available) ----------
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def _get_imagenet_mean_std():
    # prefer your get_imagenet_mean_std if defined
    if "get_imagenet_mean_std" in globals():
        try:
            m, s = get_imagenet_mean_std()
            if isinstance(m, (list, tuple)) and isinstance(s, (list, tuple)):
                return tuple(m), tuple(s)
        except Exception:
            pass
    # fallback
    try:
        meta = getattr(ResNet34_Weights.DEFAULT, "meta", {}) or {}
        return tuple(meta.get("mean", IMAGENET_MEAN)), tuple(meta.get("std", IMAGENET_STD))
    except Exception:
        return IMAGENET_MEAN, IMAGENET_STD

# ---------- 3) Utilities (reuse your compute_class_weights if present) ----------
def _compute_class_weights(counts: List[int]) -> torch.Tensor:
    if "compute_class_weights" in globals():
        try:
            t = compute_class_weights(counts)
            if isinstance(t, torch.Tensor):
                return t
        except Exception:
            pass
    total, K = sum(counts), len(counts)
    w = [total / (K * c) if c > 0 else 0.0 for c in counts]
    return torch.tensor(w, dtype=torch.float)

def _load_label_map(ckpt_path: str) -> Optional[Dict[int, str]]:
    ckpt = Path(ckpt_path)
    # candidates to look for label_map.json
    candidates = [
        ckpt.with_suffix("").with_name("label_map.json"),
        ckpt.parent / "label_map.json",
        Path("checkpoints/label_map.json"),
        Path("/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/label_map.json"), # Check Shared Drive
    ]
    for p in candidates:
        if p.exists():
            try:
                j = json.loads(p.read_text(encoding="utf-8"))
                return {int(k): v for k, v in j.items()}
            except Exception:
                pass
    return None

def _classes_from_label_map(lm: Dict[int, str]) -> List[str]:
    return [lm[i] for i in sorted(lm.keys())]

def _count_by_class(imgfolder: datasets.ImageFolder, class_order: List[str]) -> List[int]:
    idx_by_name = {name: i for i, name in enumerate(class_order)}
    counts = [0] * len(class_order)
    for path, y in imgfolder.samples:
        name = imgfolder.classes[y]
        if name in idx_by_name:
            counts[idx_by_name[name]] += 1
    return counts

class _RemapTargetsDataset(Dataset):
    """Map ImageFolder targets to a desired index order from label_map/classes."""
    def __init__(self, base: datasets.ImageFolder, name_to_desired: Dict[str, int]):
        self.base = base
        self.loader = base.loader
        self.transform = base.transform
        self.samples = []
        self.classes = [None] * len(name_to_desired)
        # invert mapping for class names list
        for name, desired in name_to_desired.items():
            if 0 <= desired < len(self.classes):
                self.classes[desired] = name
        # filter/remap samples
        for path, y in base.samples:
            name = base.classes[y]
            if name in name_to_desired:
                self.samples.append((path, name_to_desired[name]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, y = self.samples[i]
        x = self.loader(path)
        if self.transform is not None:
            x = self.transform(x)
        return x, y

# ---------- 4) Build outputs: classes, test_dl, criterion, dev ----------
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_root = Path(EVAL.data_root)
train_dir = data_root / "train"
test_dir  = data_root / "test"

if not test_dir.exists():
    raise FileNotFoundError(f"Missing folder: {test_dir} (expected test data at {test_dir}/<class>/...)")

# (a) Resolve classes (prefer label_map for consistent order with training)
label_map = _load_label_map(EVAL.ckpt_path)
if label_map is not None:
    classes = _classes_from_label_map(label_map)
else:
    # If no label_map, infer from train/ (preferred) or from test/ as fallback
    src_dir = train_dir if train_dir.exists() else test_dir
    classes = datasets.ImageFolder(src_dir).classes

# (b) Eval transforms (reuse your eval tfms if defined; else standard)
if "eval_tfms" in globals() and isinstance(eval_tfms, transforms.Compose):
    _eval_tfms = eval_tfms
else:
    mean, std = _get_imagenet_mean_std()
    _eval_tfms = transforms.Compose([
        transforms.Resize(int(EVAL.img_size * 1.15)),
        transforms.CenterCrop(EVAL.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

# (c) Test dataset (remap to classes order if label_map present)
base_test = datasets.ImageFolder(test_dir, transform=_eval_tfms)
if label_map is not None:
    name_to_idx = {label_map[i]: i for i in sorted(label_map.keys())}
    test_ds = _RemapTargetsDataset(base_test, name_to_idx)
else:
    test_ds = base_test  # assumes folder order matches training order

test_dl = DataLoader(
    test_ds, batch_size=EVAL.batch_size, shuffle=False,
    num_workers=EVAL.num_workers, pin_memory=True
)

# (d) Criterion (weighted CE using train counts if available)
if EVAL.use_class_weights and train_dir.exists():
    train_imgfolder = datasets.ImageFolder(train_dir, transform=None)
    counts = _count_by_class(train_imgfolder, classes)
    class_weights = _compute_class_weights(counts).to(dev)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

print("Device:", dev)
print("Classes:", classes)
print("Num test images:", len(test_ds))
print("Criterion:", criterion)

Device: cuda
Classes: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009']
Num test images: 980
Criterion: CrossEntropyLoss()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [19]:
import os
from pathlib import Path

# Define the best checkpoint path logic (Local vs Drive)
ckpt_name = "resnet34_faces_best_32.pt"
local_path = Path("checkpoints") / ckpt_name
drive_path = Path("/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints") / ckpt_name

if local_path.exists():
    best_ckpt_path = str(local_path)
    print(f"Using local checkpoint: {best_ckpt_path}")
elif drive_path.exists():
    best_ckpt_path = str(drive_path)
    print(f"Using Drive checkpoint: {best_ckpt_path}")
else:
    # Fallback to local path if neither exists (will fail later, but consistent)
    best_ckpt_path = str(local_path)
    print(f"Warning: Checkpoint not found at {local_path} or {drive_path}. Defaulting to local.")

Using local checkpoint: checkpoints/resnet34_faces_best_32.pt


In [20]:
# Rebuild test_dl with a single worker to avoid multiprocessing issues on Windows/Jupyter
from torch.utils.data import DataLoader
test_dl = DataLoader(
    test_ds,
    batch_size=EVAL.batch_size,
    shuffle=False,
    num_workers=0,                # <— key change
    pin_memory=True,
)


In [21]:
# === Cell: Load saved model (.pkl) and evaluate on test_dl ===
from torchvision import models

# Removed _build_resnet50 as we have a generic build_model function

def load_model_generic(ckpt_path: str, num_classes: int, device: torch.device):
    obj = torch.load(ckpt_path, map_location=device)

    # Use the existing build_model function which creates the correct ResNet34 architecture
    model = build_model(num_classes)

    if isinstance(obj, torch.nn.Module): # This case is less likely with our current save format
        model.load_state_dict(obj.state_dict())
    elif isinstance(obj, dict) and "model_state" in obj:
        model.load_state_dict(obj["model_state"])
    else:
        # assume state_dict directly
        model.load_state_dict(obj)

    model.to(device)
    model.eval()
    return model

@torch.no_grad()
def eval_on_loader(model, loader, device):
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        total_correct += (pred == y).sum().item()
        total += x.size(0)
    return total_loss / max(total, 1), total_correct / max(total, 1)

# set your checkpoint path here if different
CKPT_PATH = EVAL.ckpt_path
model = load_model_generic(best_ckpt_path, num_classes=len(classes), device=dev)
test_loss, test_acc = eval_on_loader(model, test_dl, dev)
print(f"[TEST] loss={test_loss:.4f} acc={test_acc:.3f}")


[TEST] loss=0.8834 acc=0.768


In [22]:
try:
        from sklearn.metrics import confusion_matrix, classification_report
        y_true, y_pred = [], []
        model.eval()
        with torch.no_grad():
            for images, targets in test_dl:
                images = images.to(dev, non_blocking=True)
                logits = model(images).cpu()
                preds = logits.argmax(1)
                y_true.extend(targets.numpy().tolist())
                y_pred.extend(preds.numpy().tolist())
        cm = confusion_matrix(y_true, y_pred)
        print("Confusion matrix:\n", cm)
        print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=classes))
except Exception as e:
        print("Skipped detailed test report (sklearn not available or error):", e)



Confusion matrix:
 [[77  2  4  3  0  2  2  7  2  1]
 [ 1 87  1  0  0  2  0  4  2  0]
 [ 7  6 68  2  0  5  0  3  7  0]
 [ 2  3  2 70  1  2 13  2  0  3]
 [ 4  1  0  9 68  7  6  0  1  4]
 [ 0  0  0  2 10 66  4  1  2  8]
 [ 1  0  0  3  0  6 84  0  1  3]
 [ 9  6  7  3  0  0  0 67  6  2]
 [ 3  3  0  1  0  2  1  2 84  2]
 [ 1  0  0  3  1  5  2  4  0 82]]

Classification report:
               precision    recall  f1-score   support

        0000       0.73      0.77      0.75       100
        0001       0.81      0.90      0.85        97
        0002       0.83      0.69      0.76        98
        0003       0.73      0.71      0.72        98
        0004       0.85      0.68      0.76       100
        0005       0.68      0.71      0.69        93
        0006       0.75      0.86      0.80        98
        0007       0.74      0.67      0.71       100
        0008       0.80      0.86      0.83        98
        0009       0.78      0.84      0.81        98

    accuracy                 

In [23]:
import shutil
from pathlib import Path

# Source paths (local)
ckpt_name = "resnet34_faces_best_32.pt"
local_ckpt = Path("checkpoints") / ckpt_name
local_label_map = Path("checkpoints/label_map.json")

# Destination paths (Shared Drive)
dest_dir = Path("/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints")
dest_dir.mkdir(parents=True, exist_ok=True)

# Copy Model
if local_ckpt.exists():
    shutil.copy(local_ckpt, dest_dir / ckpt_name)
    print(f"Successfully copied model to: {dest_dir / ckpt_name}")
else:
    print(f"Error: Local model not found at {local_ckpt}")

# Copy Label Map
if local_label_map.exists():
    shutil.copy(local_label_map, dest_dir / "label_map.json")
    print(f"Successfully copied label map to: {dest_dir / 'label_map.json'}")
else:
    print(f"Warning: Local label map not found at {local_label_map}")

Successfully copied model to: /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet34_faces_best_32.pt
Successfully copied label map to: /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/label_map.json
